# NLP-20: Khảo Sát Thống Kê Dữ Liệu Tóm Tắt Tiếng Việt (EDA)
### Nhóm 10 - Khoa Công nghệ Thông tin - Trường Đại học Kinh tế - Kỹ thuật Công nghiệp (UNETI)

Notebook này thực hiện các bước khảo sát dữ liệu chuẩn mực theo yêu cầu mục 7 và 8 của Đề tài:
1. Kiểm tra tính hợp lệ của dữ liệu theo `schema.json`.
2. Thống kê phân bố độ dài văn bản gốc (document) và tóm tắt (summary).
3. Phân tích tỷ lệ nén (compression ratio) và độ phong phú từ vựng (TTR).
4. Khảo sát trùng lặp và phân bố các miền lĩnh vực (Domains) cũng như các trường hợp biên (Edge Cases).

In [ ]:
import os
import sys
import json
import numpy as np

# Thiết lập thư mục gốc dự án
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl, load_json
from src.preprocessor import VietnamesePreprocessor
from src.data_loader import validate_sample_schema

## 1. Nạp và kiểm tra Schema dữ liệu

In [ ]:
data_path = os.path.join(PROJECT_ROOT, "data", "sample_benchmark.jsonl")
samples = load_jsonl(data_path)
print(f"Tổng số mẫu nạp thành công: {len(samples)}")

# Kiểm tra schema cho từng mẫu
valid_count = 0
for s in samples:
    is_valid, msg = validate_sample_schema(s)
    if is_valid:
        valid_count += 1
print(f"Số mẫu thỏa mãn Schema tuyệt đối: {valid_count}/{len(samples)} ({valid_count/len(samples)*100:.1f}%)")

## 2. Thống kê Phân Bố Độ Dài và Tỷ Lệ Nén

In [ ]:
doc_lens = [len(VietnamesePreprocessor.tokenize_syllables(s['document'])) for s in samples]
sum_lens = [len(VietnamesePreprocessor.tokenize_syllables(s['summary'])) for s in samples]
ratios = [s / d for s, d in zip(sum_lens, doc_lens)]

print("=== BẢNG THỐNG KÊ ĐỘ DÀI VĂN BẢN (ĐƠN VỊ: TỪ / ÂM TIẾT) ===")
print(f"Văn bản gốc: Trung bình={np.mean(doc_lens):.1f} | Min={np.min(doc_lens)} | Max={np.max(doc_lens)} | Std={np.std(doc_lens):.1f}")
print(f"Tóm tắt vàng: Trung bình={np.mean(sum_lens):.1f} | Min={np.min(sum_lens)} | Max={np.max(sum_lens)} | Std={np.std(sum_lens):.1f}")
print(f"Tỷ lệ nén (Compression ratio): Trung bình={np.mean(ratios)*100:.2f}% | Min={np.min(ratios)*100:.2f}% | Max={np.max(ratios)*100:.2f}%")

## 3. Phân bố theo Lĩnh Vực (Domain) và Trường Hợp Biên (Edge Cases)

In [ ]:
from collections import Counter
domains = Counter([s.get('domain', 'Không rõ') for s in samples])
print("Phân bố lĩnh vực bài báo:")
for d, cnt in domains.items():
    print(f"  - {d:<25}: {cnt} mẫu ({cnt/len(samples)*100:.1f}%)")